# SBV2 04 - OCR Baseline

This notebook runs the first local OCR baseline over the unified Sandbox V2 manifest from Notebook 03. It uses PaddleOCR with cached PP-OCRv5 English detection/recognition models, then saves line text, boxes, document summaries, quick metrics and visual overlays.

Default run is intentionally tiny: one document per source, up to eight documents. Raise the controls after checking the first overlays.

## Contract

Input:

- `data/interim/sandbox_v2/manifests/sbv2_03_document_manifest.csv`

Outputs:

- `data/interim/sandbox_v2/manifests/sbv2_04_ocr_sample_manifest.csv`
- `data/interim/sandbox_v2/ocr_outputs/sbv2_04_ocr_document_outputs.jsonl`
- `data/interim/sandbox_v2/ocr_outputs/sbv2_04_ocr_text_regions.csv`
- OCR audit tables under `outputs/sandbox_v2/audit_tables/`
- OCR overlay contact sheet under `outputs/figures/sandbox_v2/`

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "sandbox_v2").exists():
    candidates = [parent for parent in ROOT.parents if (parent / "sandbox_v2").exists()]
    if candidates:
        ROOT = candidates[0]
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

CACHE_DIR = Path(os.environ.get("MENUFORGE_CACHE_HOME", ROOT / ".cache")).resolve()
for env_name, relative_path in {
    "XDG_CACHE_HOME": "",
    "PADDLE_PDX_CACHE_HOME": "paddlex",
    "MPLCONFIGDIR": "matplotlib",
    "JUPYTER_CONFIG_DIR": "jupyter_config",
    "JUPYTER_DATA_DIR": "jupyter_data",
    "JUPYTER_RUNTIME_DIR": "jupyter_runtime",
    "IPYTHONDIR": "ipython",
}.items():
    target = CACHE_DIR / relative_path if relative_path else CACHE_DIR
    target.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault(env_name, str(target))

os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
pd.set_option("display.max_columns", 90)
pd.set_option("display.max_colwidth", 180)
print(f"Project root: {ROOT}")

Project root: /home/endy/menuforge


In [2]:
# Keep this tiny while we inspect OCR behavior. Raise these after the overlays look sane.
RUN_OCR = os.environ.get("SBV2_RUN_OCR", "1") == "1"
MAX_PER_SOURCE = int(os.environ.get("SBV2_OCR_MAX_PER_SOURCE", "1"))
MAX_DOCS_TEXT = os.environ.get("SBV2_OCR_MAX_DOCS", "8").strip()
MAX_DOCS = int(MAX_DOCS_TEXT) if MAX_DOCS_TEXT else None

print({"RUN_OCR": RUN_OCR, "MAX_PER_SOURCE": MAX_PER_SOURCE, "MAX_DOCS": MAX_DOCS})

{'RUN_OCR': True, 'MAX_PER_SOURCE': 1, 'MAX_DOCS': 8}


In [3]:
from sandbox_v2.ocr_baseline import write_outputs

summary_path = ROOT / "outputs" / "sandbox_v2" / "audit_tables" / "sbv2_04_ocr_summary.json"
if RUN_OCR:
    summary = write_outputs(max_per_source=MAX_PER_SOURCE, max_docs=MAX_DOCS)
elif summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
else:
    raise FileNotFoundError("RUN_OCR is false and no previous SBV2 04 summary exists.")

print(json.dumps({
    "ocr_engine": summary.get("ocr_engine"),
    "engine_status": summary.get("engine_status"),
    "sample_rows": summary.get("sample_rows"),
    "ok_rows": summary.get("ok_rows"),
    "failure_rows": summary.get("failure_rows"),
    "text_region_rows": summary.get("text_region_rows"),
    "mean_confidence": summary.get("mean_confidence"),
    "mean_runtime_seconds": summary.get("mean_runtime_seconds"),
}, indent=2))

/home/endy/menuforge/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/endy/menuforge/.cache/paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/endy/menuforge/.cache/paddlex/official_models/en_PP-OCRv5_mobile_rec`.
Resized image size (8160x6144) exceeds max_side_limit of 4000. Resizing to fit within limit.


{
  "ocr_engine": "paddleocr_ppocrv5_en_cpu",
  "engine_status": "ready",
  "sample_rows": 8,
  "ok_rows": 8,
  "failure_rows": 0,
  "text_region_rows": 1485,
  "mean_confidence": 0.9413,
  "mean_runtime_seconds": 9.977
}


In [4]:
source_summary = pd.read_csv(ROOT / summary["source_summary_csv"])
metric_preview = pd.read_csv(ROOT / summary["metric_preview_csv"])
failures = pd.read_csv(ROOT / summary["failures_csv"])
regions = pd.read_csv(ROOT / summary["text_regions_csv"])

display(Markdown("### Source Summary"))
display(source_summary)
display(Markdown("### Metric Preview"))
display(metric_preview.head(20))
display(Markdown("### Text Regions Preview"))
display(regions.head(20))

if failures.empty:
    display(Markdown("### Failures\n\nNo OCR failures in this run."))
else:
    display(Markdown("### Failures"))
    display(failures)

EmptyDataError: No columns to parse from file

In [ ]:
contact_sheet = summary.get("overlay_contact_sheet")
if contact_sheet:
    display(Markdown("### OCR Overlay Contact Sheet"))
    display(Image(filename=str(ROOT / contact_sheet)))
else:
    display(Markdown("No OCR overlay contact sheet was created."))

## Handoff To Notebook 05

Notebook 05 should read `data/interim/sandbox_v2/ocr_outputs/sbv2_04_ocr_text_regions.csv` and `data/interim/sandbox_v2/ocr_outputs/sbv2_04_ocr_document_outputs.jsonl` as the OCR baseline context. The VLM/document parser should be compared against these rows, not against visual impressions alone.